<a href="https://colab.research.google.com/github/NielsRogge/Transformers-Tutorials/blob/master/DETR/Fine_tuning_DetrForObjectDetection_on_custom_dataset_(balloon).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Fine-tuning DETR on a custom dataset for object detection

In this notebook, we are going to fine-tune [DETR](https://huggingface.co/docs/transformers/model_doc/detr) (end-to-end object detection with Transformers) on a custom object detection dataset. The goal for the model is to detect balloons in pictures.

* Original DETR paper: https://arxiv.org/abs/2005.12872
* Original DETR repo: https://github.com/facebookresearch/detr

### Note regarding GPU memory

`DetrImageProcessor` by default resizes each image to have a `min_size` of 800 pixels and a `max_size` of 1333 pixels (as these are the default values that DETR uses at *inference* time). Note that this can stress-test the memory of your GPU when *training* the model, as the images are flattened after being sent through the convolutional backbone. The sequence length that is sent through the Transformer is typically of length `(height * width / 32^2)`. So if you consider an image of size `(900, 900)` for example, the sequence length is `900^2 / 32^2 = 791`, which is larger than what NLP models like BERT use (512). It's advised to use a batch size of 2 on a single GPU. You can of course also initialize `DetrImageProcessor` with a smaller `size` to use bigger batches.

### Note regarding data augmentation

DETR actually uses several image augmentations during training. One of them is **scale augmentation**: they set the `min_size` randomly to be one of [480, 512, 544, 576, 608, 640, 672, 704, 736, 768, 800] as can be seen [here](https://github.com/facebookresearch/detr/blob/a54b77800eb8e64e3ad0d8237789fcbf2f8350c5/datasets/coco.py#L122). However, we are not going to add any of the augmentations that are used in the original implementation during training. It works fine without them.

### Training framework

We're going to fine-tune the model using [PyTorch Lightning](https://lightning.ai/docs/pytorch/stable/), but of course you could also train the model using native PyTorch, the 🤗 [Trainer](https://huggingface.co/docs/transformers/main_classes/trainer) class, 🤗 [Accelerate](https://huggingface.co/docs/accelerate/index), or any other framework you prefer.

Also big thanks to the creator of [this notebook](https://github.com/woctezuma/finetune-detr/blob/master/finetune_detr.ipynb), which helped me a lot in understanding how to fine-tune DETR on a custom dataset.


## Set up environment

In [ ]:
%pip install -q transformers torch torchvision pytorch-lightning timm datasets pycocotools pillow

## Load dataset

To load your custom dataset into an object detection one, I recommend [this guide](https://huggingface.co/docs/datasets/en/image_dataset#object-detection).

Here we will load the [balloon-dataset](https://huggingface.co/datasets/nielsr/balloon-dataset) which already follows the Hugging Face object-detection `imagefolder` layout.

In [ ]:
from datasets import load_dataset

balloon = load_dataset("nielsr/balloon-dataset")
balloon

In [ ]:
print(balloon["train"].column_names)
print(balloon["train"][0]["objects"].keys())

In [ ]:
id2label = {0: "balloon"}
label2id = {label: idx for idx, label in id2label.items()}

## Create PyTorch dataset + dataloaders

The Hub dataset already stores the annotations in a Datasets-compatible object-detection format. We wrap it in a small PyTorch dataset that recreates the COCO-style annotations expected by `DetrImageProcessor` and by the COCO evaluation code later in the notebook.

In [ ]:
import os
from pycocotools.coco import COCO
from torch.utils.data import Dataset


class HuggingFaceCocoDetection(Dataset):
    def __init__(self, dataset, image_processor):
        self.dataset = dataset
        self.image_processor = image_processor
        self.ids = list(dataset["image_id"])
        self.id2index = {image_id: idx for idx, image_id in enumerate(self.ids)}
        self.coco = COCO()
        self.coco.dataset = self._build_coco_dataset()
        self.coco.createIndex()

    def _build_coco_dataset(self):
        images = []
        annotations = []

        for example in self.dataset:
            image = example["image"]
            image_id = example["image_id"]
            images.append(
                {
                    "id": image_id,
                    "file_name": os.path.basename(image.filename),
                    "width": example["width"],
                    "height": example["height"],
                }
            )

            for annotation_id, bbox, category_id, area in zip(
                example["objects"]["id"],
                example["objects"]["bbox"],
                example["objects"]["categories"],
                example["objects"]["area"],
            ):
                annotations.append(
                    {
                        "id": annotation_id,
                        "image_id": image_id,
                        "category_id": category_id,
                        "bbox": bbox,
                        "area": area,
                        "iscrowd": 0,
                    }
                )

        return {
            "images": images,
            "annotations": annotations,
            "categories": [{"id": 0, "name": "balloon", "supercategory": "object"}],
        }

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        example = self.dataset[idx]
        image = example["image"].convert("RGB")
        image_id = example["image_id"]
        target = {"image_id": image_id, "annotations": self.coco.imgToAnns[image_id]}
        encoding = self.image_processor(images=image, annotations=target, return_tensors="pt")
        pixel_values = encoding["pixel_values"].squeeze()
        target = encoding["labels"][0]
        return pixel_values, target

    def get_image(self, image_id):
        return self.dataset[self.id2index[image_id]]["image"].convert("RGB")


Based on the class defined above, we create training and validation datasets from the Hub splits.

In [ ]:
from transformers import DetrImageProcessor

image_processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")

train_dataset = HuggingFaceCocoDetection(dataset=balloon["train"], image_processor=image_processor)
val_dataset = HuggingFaceCocoDetection(dataset=balloon["validation"], image_processor=image_processor)

As you can see, this dataset is tiny:

In [ ]:
print("Number of training examples:", len(train_dataset))
print("Number of validation examples:", len(val_dataset))

Let's verify an example by visualizing it. We can still access the reconstructed COCO API through `train_dataset.coco`.

In [ ]:
import numpy as np
from PIL import ImageDraw

image_ids = train_dataset.coco.getImgIds()
image_id = image_ids[np.random.randint(0, len(image_ids))]
print("Image n°{}".format(image_id))
image = train_dataset.get_image(image_id)

annotations = train_dataset.coco.imgToAnns[image_id]
draw = ImageDraw.Draw(image, "RGBA")

cats = train_dataset.coco.cats
id2label = {k: v["name"] for k, v in cats.items()}

for annotation in annotations:
    box = annotation["bbox"]
    class_idx = annotation["category_id"]
    x, y, w, h = tuple(box)
    draw.rectangle((x, y, x + w, y + h), outline="red", width=1)
    draw.text((x, y), id2label[class_idx], fill="white")

image

Next, let's create corresponding PyTorch dataloaders. We define a custom `collate_fn` to batch images together. As DETR resizes images to have a min size of 800 and a max size of 1333, images can have different sizes. We pad images (`pixel_values`) to the largest image in a batch, and create a corresponding `pixel_mask` to indicate which pixels are real (1) / which are padding (0).

In [ ]:
from torch.utils.data import DataLoader

def collate_fn(batch):
    pixel_values = [item[0] for item in batch]
    encoding = image_processor.pad(pixel_values, return_tensors="pt")
    labels = [item[1] for item in batch]
    return {
        "pixel_values": encoding["pixel_values"],
        "pixel_mask": encoding["pixel_mask"],
        "labels": labels,
    }

train_dataloader = DataLoader(train_dataset, collate_fn=collate_fn, batch_size=2, shuffle=True)
val_dataloader = DataLoader(val_dataset, collate_fn=collate_fn, batch_size=2)
batch = next(iter(train_dataloader))
batch.keys()

Let's verify the shape of the `pixel_values`, and check the `target`:

In [ ]:
pixel_values, target = train_dataset[0]
print(pixel_values.shape)
print(target)

## Train the model using PyTorch Lightning

Here we define a `LightningModule`, which is an `nn.Module` with some extra functionality.

For more information regarding PyTorch Lightning, I recommend the [docs](https://lightning.ai/docs/pytorch/stable/) as well as the [tutorial notebooks](https://github.com/Lightning-AI/tutorials).

You can of course just train the model in native PyTorch as an alternative.

In [ ]:
import pytorch_lightning as pl
from transformers import DetrForObjectDetection
import torch


class Detr(pl.LightningModule):
    def __init__(self, lr, lr_backbone, weight_decay):
        super().__init__()
        # replace COCO classification head with custom head
        # we specify the "no_timm" variant here to not rely on the timm library
        # for the convolutional backbone
        self.model = DetrForObjectDetection.from_pretrained(
            "facebook/detr-resnet-50",
            revision="no_timm",
            num_labels=len(id2label),
            ignore_mismatched_sizes=True,
        )
        self.lr = lr
        self.lr_backbone = lr_backbone
        self.weight_decay = weight_decay

    def forward(self, pixel_values, pixel_mask):
        return self.model(pixel_values=pixel_values, pixel_mask=pixel_mask)

    def common_step(self, batch, batch_idx):
        pixel_values = batch["pixel_values"]
        pixel_mask = batch["pixel_mask"]
        labels = [{k: v.to(self.device) for k, v in t.items()} for t in batch["labels"]]
        outputs = self.model(pixel_values=pixel_values, pixel_mask=pixel_mask, labels=labels)
        return outputs.loss, outputs.loss_dict

    def training_step(self, batch, batch_idx):
        loss, loss_dict = self.common_step(batch, batch_idx)
        self.log("training_loss", loss)
        for k, v in loss_dict.items():
            self.log("train_" + k, v.item())
        return loss

    def validation_step(self, batch, batch_idx):
        loss, loss_dict = self.common_step(batch, batch_idx)
        self.log("validation_loss", loss)
        for k, v in loss_dict.items():
            self.log("validation_" + k, v.item())
        return loss

    def configure_optimizers(self):
        param_dicts = [
            {"params": [p for n, p in self.named_parameters() if "backbone" not in n and p.requires_grad]},
            {
                "params": [p for n, p in self.named_parameters() if "backbone" in n and p.requires_grad],
                "lr": self.lr_backbone,
            },
        ]
        return torch.optim.AdamW(param_dicts, lr=self.lr, weight_decay=self.weight_decay)

    def train_dataloader(self):
        return train_dataloader

    def val_dataloader(self):
        return val_dataloader


As PyTorch Lightning by default logs to Tensorboard, let's start it:

In [ ]:
# Start tensorboard when the extension is available.
try:
    get_ipython().run_line_magic("load_ext", "tensorboard")
    get_ipython().run_line_magic("tensorboard", "--logdir lightning_logs/")
except Exception as error:
    print(f"TensorBoard is unavailable in this environment: {error}")


Here we define the model, and verify the outputs.

In [ ]:
model = Detr(lr=1e-4, lr_backbone=1e-5, weight_decay=1e-4)
outputs = model(pixel_values=batch["pixel_values"], pixel_mask=batch["pixel_mask"])
outputs.logits.shape

The logits are of shape `(batch_size, num_queries, number of classes + 1)`. The model internally adds an additional "no object class", which explains why we have one additional output for the class dimension.

Next, let's train! We train for a maximum of 300 training steps, and also use gradient clipping. You can refresh Tensorboard above to check the various losses.

In [ ]:
from pytorch_lightning import Trainer

trainer = Trainer(accelerator="auto", max_steps=300, gradient_clip_val=0.1)
trainer.fit(model)

## Push to the hub

We can simply call `push_to_hub` on our model and image processor after training to upload them to the 🤗 hub. Note that you can pass `private=True` if you don't want to share the model with the world (keep the model private).

In [ ]:
from huggingface_hub import login

# login()  # uncomment to authenticate before pushing

In [ ]:
# model.model.push_to_hub("nielsr/detr-finetuned-balloon-v2")
# image_processor.push_to_hub("nielsr/detr-finetuned-balloon-v2")

We can easily reload a trained checkpoint, and move it to the GPU as follows:

In [ ]:
from transformers import DetrImageProcessor, DetrForObjectDetection
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
trained_model = model.model.to(device)
trained_model.eval()
# Or load from the hub:
# trained_model = DetrForObjectDetection.from_pretrained("nielsr/detr-finetuned-balloon-v2").to(device)
# image_processor = DetrImageProcessor.from_pretrained("nielsr/detr-finetuned-balloon-v2")

## Evaluate the model

Finally, we evaluate the model on the validation set with `pycocotools`.

In [ ]:
from pycocotools.cocoeval import COCOeval
from tqdm.notebook import tqdm
import numpy as np

print("Running evaluation...")
all_results = []

for batch in tqdm(val_dataloader):
    pixel_values = batch["pixel_values"].to(device)
    pixel_mask = batch["pixel_mask"].to(device)
    labels = [{k: v.to(device) for k, v in t.items()} for t in batch["labels"]]

    with torch.no_grad():
        outputs = trained_model(pixel_values=pixel_values, pixel_mask=pixel_mask)

    orig_target_sizes = torch.stack([target["orig_size"] for target in labels], dim=0)
    results = image_processor.post_process_object_detection(
        outputs, target_sizes=orig_target_sizes, threshold=0.0
    )

    for target, result in zip(labels, results):
        image_id = target["image_id"].item()
        boxes = result["boxes"].cpu().numpy()
        scores = result["scores"].cpu().numpy()
        labels_pred = result["labels"].cpu().numpy()

        boxes_xywh = boxes.copy()
        boxes_xywh[:, 2] = boxes[:, 2] - boxes[:, 0]
        boxes_xywh[:, 3] = boxes[:, 3] - boxes[:, 1]

        for box, score, label in zip(boxes_xywh, scores, labels_pred):
            all_results.append(
                {
                    "image_id": image_id,
                    "category_id": int(label),
                    "bbox": box.tolist(),
                    "score": float(score),
                }
            )

coco_gt = val_dataset.coco
coco_dt = coco_gt.loadRes(all_results)
coco_eval = COCOeval(coco_gt, coco_dt, "bbox")
coco_eval.evaluate()
coco_eval.accumulate()
coco_eval.summarize()

## Inference (+ visualization)

Let's visualize the predictions of DETR on the first image of the validation set.

In [ ]:
import matplotlib.pyplot as plt

# colors for visualization
COLORS = [[0.000, 0.447, 0.741], [0.850, 0.325, 0.098], [0.929, 0.694, 0.125],
          [0.494, 0.184, 0.556], [0.466, 0.674, 0.188], [0.301, 0.745, 0.933]]


def plot_results(pil_img, scores, labels, boxes, id2label):
    plt.figure(figsize=(16, 10))
    plt.imshow(pil_img)
    ax = plt.gca()
    colors = COLORS * 100
    for score, label, (xmin, ymin, xmax, ymax), c in zip(
        scores.tolist(), labels.tolist(), boxes.tolist(), colors
    ):
        ax.add_patch(plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,
                                   fill=False, color=c, linewidth=3))
        text = f"{id2label[label]}: {score:0.2f}"
        ax.text(xmin, ymin, text, fontsize=15,
                bbox=dict(facecolor="yellow", alpha=0.5))
    plt.axis("off")
    plt.show()


In [ ]:
pixel_values, target = val_dataset[1]
pixel_values = pixel_values.unsqueeze(0).to(device)

with torch.no_grad():
    outputs = trained_model(pixel_values=pixel_values, pixel_mask=None)

image_id = target["image_id"].item()
image = val_dataset.get_image(image_id)
width, height = image.size
results = image_processor.post_process_object_detection(
    outputs, target_sizes=[(height, width)], threshold=0.9
)[0]
plot_results(image, results["scores"].cpu(), results["labels"].cpu(), results["boxes"].cpu(), id2label)